In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

from extremal_mil import fig2_experiment, ode, plotting, theory

plt.rcParams.update(plotting.RCPARAMS)

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

%config InlineBackend.figure_format = 'svg'

In [ ]:
# list on disk, as (kind, ell, d, lr, n_steps) 
available = fig2_experiment.list_trajectory_sweeps()
print(f"{len(available)} trajectory-sweep cells in {fig2_experiment.DEFAULT_DATA_DIR}")
for kind in ("u", "rho"):
    for ell in fig2_experiment.ELL_VALUES_TRAJ:
        for lr, ns in fig2_experiment.LR_STEPS_TRAJ:
            ds = [c[2] for c in available if c[0] == kind and c[1] == ell
                  and np.isclose(c[3], lr) and c[4] == ns]
            print(f"  {kind:>3}  ell={ell:<5d} lr={lr:<5g} s={ns:<5d} d={ds}")

In [ ]:
# The cells to plot. KIND selects which sweep's runs are drawn; the phase-plane
# rows then show those same runs, projected onto (u, lambda) and (rho, lambda).
KIND = "u"
ELL = 100
D_SHOW = [22, 150]

# lr=0.1: the (rho, lambda) path is visibly jagged at lr=0.3, still is at 0.2.
SCHEDULE = (0.1, 150000)


def load_cell(d):
    # One schedule, no fallback. Falling back to another lr or horizon when a
    # cell is missing silently changes the data the figure shows, which is how
    # the panels came to be drawn from different ell.
    lr, n_steps = SCHEDULE
    return fig2_experiment.load_trajectory_sweep(KIND, ELL, d, lr, n_steps)


alpha, rho_true, q = fig2_experiment.ALPHA, fig2_experiment.RHO_TRUE, fig2_experiment.Q

traj = {d: load_cell(d) for d in D_SHOW}
x0 = traj[D_SHOW[0]]["x0"]
T_MAX = max(float(t["flow_time"]) for t in traj.values())
colors, mappable = plotting.run_colors(x0)
CBAR_LABEL = "$u_0$" if KIND == "u" else r"$\rho_0$"

# Loss-row limits, set by the range GD occupies -- see plotting.finish_grid.
LOSS_LIM = (min(t["loss"].min() for t in traj.values()) / 3,
            max(t["loss"].max() for t in traj.values()) * 3)

for d, t in traj.items():
    print(f"d={d:<4d} n={int(t['n']):<6d} alpha_eff={np.log(int(t['n'])) / d:.4f}  "
          f"lr={float(t['lr']):<5g} steps={int(t['n_steps']):<6d} "
          f"t up to {float(t['flow_time']):g}")

In [ ]:
# The flow is drawn only in the phase planes, as a path rather than against
# time, and it reaches its fixed point well before t = 300. Integrating it
# over the trajectories' full horizon instead (t = 15000, at max_step = 0.01)
# is ~1.5M steps per initial condition and does not change the drawn curve.
FLOW_T_MAX = min(T_MAX, 300.0)

sols = [ode.integrate([fig2_experiment.LAM0, u0, fig2_experiment.RHO0],
                      [0, FLOW_T_MAX], alpha, rho_true, q, max_step=0.05,
                      t_eval=np.linspace(0, FLOW_T_MAX, 601))
        for u0 in x0]
flow = {"t": sols[0].t}
flow["lam"], flow["u"], flow["rho"] = (np.vstack([s.y[k] for s in sols]) for k in range(3))
flow["loss"] = np.array([[theory.loss_asymp(l, u, r, alpha, rho_true, q)
                          for l, u, r in zip(*row)]
                         for row in zip(flow["lam"], flow["u"], flow["rho"])])
drift = np.abs(flow["u"][:, -1] - flow["u"][:, -50]).max()
print(f"{len(sols)} flow trajectories, t in [0, {flow['t'][-1]:g}]; "
      f"max |du| over the last 50 points = {drift:.1e} (settled)")

In [ ]:
d_plot = D_SHOW
fig,axes = plt.subplots(figsize=(plotting.COL_W, 3.8),nrows = 2,ncols=len(d_plot),sharey="row", sharex="row", squeeze=False,
                        layout="constrained")

for col, d in enumerate(d_plot):
    t = traj[d]
    #plotting.draw_backdrop(axes[:, col], t["t"], t["lam"], t["u"], t["rho"], t["loss"])
    # plotting.draw_runs(axes[:, col], flow["t"], flow["lam"], flow["u"], flow["rho"],
    #                    flow["loss"], colors, ends=True, lw=1.5, zorder=4,
    #                    solid_capstyle="round",plot_loss=False)

    plotting.draw_runs(axes[:, col], t["t"], t["lam"], t["u"], t["rho"], t["loss"],
                       end_marker="s",
                       colors=colors, alpha=0.6, lw=0.8, zorder=0, solid_capstyle="round",plot_loss=False)
    axes[0, col].set_title(rf"$d={d}$,  $n={int(t['n'])}$", pad=5)

plotting.finish_grid(
    fig, axes, mappable, T_MAX, LOSS_LIM,
    None, CBAR_LABEL,
    handles=plotting.endpoint_handles("s"),plot_loss=False)

# Diagonal guide lambda = lambda_star * u, clipped to the HR wedge: it runs from
# where it crosses the R/N boundary (rho_true*u = phi_alpha(lambda)) up to where
# it meets the H/L boundary at lambda = lambda_star (u=1).
lam_star = theory.lambda_star(alpha)
u_hr = brentq(lambda u: rho_true * u - theory.phi_alpha(lam_star * u, alpha), 1e-6, 1.0)

[axes[0, i].plot([u_hr, 1], [lam_star * u_hr, lam_star], "k--", lw=0.9) for i in range(len(d_plot))]
[axes[0, i].semilogy() for i in range(len(d_plot))]
[axes[1, i].semilogy() for i in range(len(d_plot))]
# After finish_grid, so the regions and their names are placed against the
# final limits and reach the panel edges rather than stopping where the

# Second-row guide lambda = sqrt(2*alpha)*rho_true/rho, clipped to start where
# it meets the H/L boundary at lambda = lambda_star (rho = rho_true), out to
# the panel's right edge.
rho_k = lam_star * rho_true
rho_h = rho_k / lam_star
rho_grid = np.linspace(rho_h, axes[1, 0].get_xlim()[1], 200)
[axes[1, i].plot(rho_grid, rho_k / rho_grid, "k--", lw=0.9) for i in range(len(d_plot))]


# regions are named once.
for col in range(len(d_plot)):
    plotting.draw_boundaries(axes[:, col], alpha, rho_true)
# axes[0, -1], not [0, 2]: the names belong on the last column whatever
# the number of d values plotted.
plotting.phase_labels(axes[0, -1], alpha, rho_true, margin=0.07, bottom_margin=0.02)
plotting.temperature_labels(axes[1, 0], alpha,pad=0.3)
fig.savefig(FIG_DIR / "fig3.pdf", bbox_inches="tight")
plt.show()